In [2]:
import TwoPUtils
%load_ext autoreload
%autoreload 2
from reward_relative.path_dict_seahorse import path_dictionary as path_dict
from reward_relative import utilities as ut
from reward_relative import plotUtils as pt
from reward_relative import spatial
from reward_relative import placeCellPlot
from reward_relative import dayData as dd
from reward_relative import behavior
import pickle
import dill
import numpy as np
import os 
import matplotlib.pyplot as plt
import TwoPUtils
import protter_functions as pf
from scipy.signal import medfilt
from matplotlib.lines import Line2D
import pandas as pd

import importlib
import h5py
import traceback
import sys
sys.path.append('/home/sosalab/local_repos/InVivoDA_analyses') 

In [3]:

experiment = 'MetaLearn'
year = 'combined'
exp_days = [3, 5, 7, 8, 10, 12, 14]

max_anim_list = dd.max_anim_list(experiment, exp_days, year=year)

## These parameters were used for computing the saved multiDayData
# bin_size = 10  # for quantifying distribution of place field peak locations
# sigma = 1  # for smoothing
# smooth = False  # whether to smooth for finding place cell peaks
# exclude_int = True  # exclude putative interneurons
# int_thresh = 0.5

## Place cell logical definitions:
## 'and' = must have significant spatial information
## in trial set 0 AND trial set 1 (i.e. before and after the reward switch)
## 'or' = must have signitive spatial information in trial set 0 OR trial set 1

# place_cell_logical = 'or'
ts_key = 'dff'  # which timeseries to use for finding peaks
# use_speed_thr = True  # use a speed threshold to calculate new trial matrices
# # speed threshold in cm/s (excludes data at speed less than this)
# speed_thr = 2

reward_dist_inclusive = 50  # in cm

# datetime of saved file
dt = "202504"

pkl_name = "%s_expdays%s_multiDayData_%s_%s.pickle" % (
    # ut.make_anim_tag(max_anim_list),
    f'm{ut.get_mouse_number(max_anim_list[0])}-{ut.get_mouse_number(max_anim_list[-1])}',
    ut.make_day_tag(
        exp_days),
    ts_key,
    dt)
pkl_path = os.path.join(
    path_dict['preprocessed_root'], 'multi_anim_sess', pkl_name)
print(pkl_path)
multiDayData = dill.load(open(pkl_path, "rb"), ignore=True)

include_ans = multiDayData[exp_days[-1]
                           ].circ_rel_stats_across_an['include_ans']
max_anim_list = sorted(np.unique(np.concatenate([multiDayData[day].anim_list
                                                 for day in exp_days])),
                       key=len)
include_ans

/data/2p_data/multi_anim_sess/m2-19_expdays3-5-7-8-10-12-14_multiDayData_dff_202504.pickle


array(['GCAMP3', 'GCAMP4', 'GCAMP7', 'GCAMP11', 'GCAMP12', 'GCAMP13',
       'GCAMP14', 'GCAMP15', 'GCAMP17', 'GCAMP18', 'GCAMP19'], dtype='<U7')

In [4]:
pkl_path = '/data/2p_data/multi_anim_sess/m2-19_expdays1-2-3-4-5-6-7-8-9-10-11-12-13-14_multiDayData_dff_202504.pickle'

multiDayData = dill.load(open(pkl_path, "rb"), ignore=True)

In [5]:
multiDayData[1].overall_place_cell_masks.keys()

dict_keys(['GCAMP2', 'GCAMP3', 'GCAMP4', 'GCAMP5', 'GCAMP6', 'GCAMP7', 'GCAMP10', 'GCAMP12', 'GCAMP13', 'GCAMP14', 'GCAMP15', 'GCAMP17', 'GCAMP18', 'GCAMP19'])

In [6]:
for day in multiDayData.keys():
    print(multiDayData[day].overall_place_cell_masks.keys() == multiDayData[day].reward_rel_cell_ids.keys())

True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [7]:
 multiDayData[1].overall_place_cell_masks

{'GCAMP2': array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True, False,  True,  True,  True,  Tr

In [8]:
multiDayData[1].reward_rel_cell_ids

{'GCAMP2': array([  0,   4,   5,   7,   8,  11,  12,  13,  15,  19,  22,  23,  28,
         30,  34,  35,  36,  37,  39,  42,  46,  47,  48,  49,  51,  52,
         55,  59,  60,  63,  66,  70,  76,  80,  81,  82,  88,  90,  91,
         94,  96,  98, 101, 104, 107, 108, 111, 120, 124, 125, 126, 131,
        136, 139, 144, 148, 149, 152, 153, 154, 155, 156, 160, 161, 166,
        169, 170, 172, 176, 177, 179, 182, 185, 186, 191, 196, 198, 199,
        203, 210, 211, 215, 216, 217, 218, 221, 222, 225, 229, 230, 231,
        232, 236, 238, 241, 243, 248, 250, 252, 254, 256, 258, 259, 260,
        264, 265, 270, 271, 277, 280, 284, 289, 290, 291, 293, 298, 300,
        301, 302, 303, 304, 306, 309, 312, 313, 316, 320, 324, 325, 329,
        331, 333, 337, 341, 344, 347, 349, 356, 359, 361, 362, 364, 367,
        378, 381, 382, 383, 384, 385, 386, 388, 389, 394, 397, 398, 405,
        407, 412, 416, 417, 419, 425, 426, 436, 440, 443, 444, 449, 452,
        463, 467, 478, 480, 484, 486, 488

In [14]:

multiDayData[13].reward_rel_cell_ids['GCAMP14']

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  23,  25,  26,  27,  28,
        30,  31,  32,  33,  34,  35,  36,  38,  39,  40,  41,  42,  43,
        44,  46,  47,  48,  50,  51,  52,  53,  55,  56,  57,  58,  59,
        60,  61,  62,  63,  64,  65,  67,  68,  69,  70,  73,  74,  75,
        76,  78,  79,  80,  82,  83,  84,  85,  86,  87,  92,  94,  96,
        97,  98,  99, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110,
       111, 112, 115, 116, 117, 119, 120, 121, 123, 124, 125, 127, 128,
       129, 130, 131, 132, 133, 134, 135, 137, 139, 141, 142, 144, 145,
       146, 147, 148, 149, 150, 151, 156, 157, 159, 160, 161, 162, 164,
       165, 166, 167, 168, 173, 174, 177, 178, 179, 180, 181, 184, 186,
       188, 189, 191, 192, 195, 196, 197, 201, 202, 203, 204, 205, 207,
       208, 209, 211, 213, 216, 218, 219, 223, 224, 225, 226, 228, 229,
       231, 232, 233, 236, 237, 238, 239, 242, 243, 245, 246, 24

In [ ]:
file_name = '/data/dave/sosa_data_h5/all_data'







with h5py.File(file_name, "r+") as f:


    # for i, day in enumerate(sorted(exp_days)):
    for i, day in enumerate(multiDayData.keys()):
        print(day)

        
        r_inds_dict = multiDayData[day].reward_rel_cell_ids 

        for animal, r_inds in r_inds_dict.items():
            try:
                today = f[animal][str(day)] 
                cell_info = today.create_group('cell_categories')

                cell_info.create_dataset('reward_cell_indices', data = r_inds)


                pc_mask = multiDayData[day].overall_place_cell_masks[animal]
                cell_info.create_dataset('overall_place_cell_masks', data = pc_mask)
                
            except:
                print(f'error ------- {animal} {day}')
                traceback.print_exc()
        

                 



1
error ------- GCAMP2 1
error ------- GCAMP3 1
error ------- GCAMP4 1
error ------- GCAMP5 1
error ------- GCAMP6 1
error ------- GCAMP7 1
error ------- GCAMP10 1
error ------- GCAMP12 1
error ------- GCAMP13 1
error ------- GCAMP14 1
error ------- GCAMP15 1
error ------- GCAMP17 1
error ------- GCAMP18 1
error ------- GCAMP19 1
2
error ------- GCAMP2 2
error ------- GCAMP3 2
error ------- GCAMP4 2
error ------- GCAMP5 2
error ------- GCAMP6 2
error ------- GCAMP7 2
error ------- GCAMP10 2
error ------- GCAMP12 2
error ------- GCAMP13 2
error ------- GCAMP14 2
error ------- GCAMP15 2
error ------- GCAMP17 2
error ------- GCAMP18 2
error ------- GCAMP19 2
4
error ------- GCAMP2 4
error ------- GCAMP3 4
error ------- GCAMP4 4
error ------- GCAMP6 4
error ------- GCAMP7 4
error ------- GCAMP10 4
error ------- GCAMP11 4
error ------- GCAMP12 4
error ------- GCAMP13 4
error ------- GCAMP14 4
error ------- GCAMP15 4
error ------- GCAMP17 4
error ------- GCAMP18 4
error ------- GCAMP19 4
6
e

Traceback (most recent call last):
  File "/tmp/ipykernel_2452605/2329896676.py", line 21, in <module>
    today = f[animal][str(day)]
           ~~~~~~~~~^^^^^^^^^^
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "/home/sosalab/miniconda3/envs/umap_env/lib/python3.11/site-packages/h5py/_hl/group.py", line 367, in __getitem__
    oid = h5o.open(self.id, self._e(name), lapl=self._lapl)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "h5py/_objects.pyx", line 54, in h5py._objects.with_phil.wrapper
  File "h5py/_objects.pyx", line 55, in h5py._objects.with_phil.wrapper
  File "h5py/h5o.pyx", line 255, in h5py.h5o.open
KeyError: "Unable to synchronously open object (object '1' doesn't exist)"
Traceback (most recent call last):
  File "/tmp/ipykernel_2452605/2329896676.py", line 21, in <module>
    today = f[animal][str(day)]
           ~~~~~~~~~^^^^^^^^^^
  File "h5py/_

In [19]:
f.close()